<h1 align="center">
<img src="https://raw.githubusercontent.com/numbbo/coco-visualize/main/cocoviz.svg" width="400"><br>
    Tutorial
</h1>

Start by installing the required packages if you haven't already.

We will use [polars](https://pola.rs) to load our different datasets and convert them into a [DataFrame](https://docs.pola.rs/api/python/stable/reference/dataframe/index.html) like object.
You don't have to use polars, you could also use pandas if you wanted or any other library that provides a DataFrame-like object.

Because we want to load serialized `Result`s, we need to include the `parquet` option when installing cocoviz. 
This is a pretty heavyweight requirement and is entirely optional. 
If you don't plan on using cocoviz's serialization utilities, you don't have to install it.

In [ ]:
!pip install polars
!pip install 'coco-visualize[parquet]'

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

from cocoviz import rtpplot, Result, ResultSet, Indicator, ProblemDescription
from pathlib import Path

Download and unpack the demo data. 
Instead of downloading the demo data, you could also run the three `run-*.py` files on your computer to create your own demo data.

In [ ]:
def download_demo_data():
    import io
    import urllib.request
    import zipfile
    
    URL = "https://f000.backblazeb2.com/file/cocoviz-demo-data/gecco26.zip"
    
    with urllib.request.urlopen(URL) as resp:
        print("Downloading demo data...")
        data = resp.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        print("Unpacking demo data...")
        zf.extractall(".")

download_demo_data()
DATA = Path("data/")

## Reading results

As you can see, the file names of all the results we are about to load have a similar structure that encodes some of the metadata we will need.
Here's a handy utility function to parse the file name and return the metadata fields as a dictionary.

In [ ]:
def parse_filename(path):
    res = dict()
    for part in path.stem.split("_"):
        key, value = part.split("=")
        res[key] = value
    return res

In order to use the results with cocoviz, we need to import them into something resembling a data frame with a column that tells cocoviz when (measured in function evaluations) some performance indicators (the rest of the columns) were observed.

We collect individual `Result`s into a `ResultSet` object.
A `ResultSet` behaves just like a Python `list`, but has some additional functions that make slicing and dicing the results a bit easier.

In [ ]:
results = ResultSet()
results

### Read Powell results from CSV files

Example of CSV file:
~~~csv
fevals,y
1,18.475384456277286
4,18.460209038138803
5,18.45554813710272
6,18.455532945374834
~~~

In [ ]:
for csv in DATA.glob("*.csv"):    
    properties = parse_filename(csv)    
    problem = ProblemDescription(properties["p"], properties["i"], number_of_variables=int(properties["d"]), number_of_objectives=1)
    df = pl.read_csv(csv)    
    results.append(Result(properties["a"], problem, df))

results

### Read CMA-ES results from JSON files

Example of JSON file:
~~~json
[
  {
    "fevals": 1,
    "y": 17.395320986065304
  },
  {
    "fevals": 2,
    "y": 15.311276285586464
  }
]
~~~

In [ ]:
for json in DATA.glob("*.json"):
    properties = parse_filename(json)
    problem = ProblemDescription(properties["p"], properties["i"], number_of_variables=int(properties["d"]), number_of_objectives=1)    
    df = pl.read_json(json)    
    results.append(Result(properties["a"], problem, df))

results

### Read native Parquet files

cocoviz can serialize `Result` objects to parquet (a binary interchange format). 
Reading these is especially easy since all the required metadata is contained in them as well.
On the other hand, they need to be specified while writing the files...

In [ ]:
for pq in DATA.glob("*.parquet"):
    results.append(Result.from_parquet(pq))

results

## Exploratory analysis

In [ ]:
results.algorithms

In [ ]:
results.problems

Make sure we have the same number of runs per problem for each algorithm.
We don't have to do this, cocoviz checks this for us where it counts, but it's a good safeguard anyway.

In [ ]:
for problem, by_problem in results.by_problem():
    n_results = [len(by_alg) for _, by_alg in by_problem.by_algorithm()]
    assert all(n == n_results[0] for n in n_results)

## Tables

If you really must, you can generate summary tables from the results using standard pandas/polars magic.

In [ ]:
rows = []
for r in results:
    rows.append({
        "algorithm": r.algorithm,
        "problem": r.problem.name,
        "instance": r.problem.instance,
        "dimension": r.problem.number_of_variables,
        "best": r["y"].min(),
    })

summary = (
    pl.DataFrame(rows)
    .group_by(["algorithm", "problem", "instance", "dimension"])
    .agg(
        pl.col("best").mean().alias("mean_best"),
        pl.col("best").std().alias("std_best"),
        pl.col("best").count().alias("n_runs"),
    )
    .with_columns(
        (pl.col("mean_best").round(4).cast(pl.Utf8) + " ± " + pl.col("std_best").round(4).cast(pl.Utf8)).alias("mean_std")
    )
    .pivot(
        on="algorithm",
        index=["problem", "instance", "dimension"],
        values="mean_std"
    )
    .sort(["problem", "instance", "dimension"])
)
summary

## Visualizing Results

If we really want to, we can still plot convergence trajectories from the results.

Note that you would probably spend more time cleaning up these plots (log axis, ...), but we are not really interested in them.

In [ ]:
for problem, by_problem in results.by_problem():
    fig, axes = plt.subplots(ncols=len(by_problem.algorithms), figsize=(12, 3), sharex=True, sharey=True)
    fig.suptitle(problem)
    for ax, (algorithm, by_problem_algorithm) in zip(axes, by_problem.by_algorithm()):        
        ax.set_title(algorithm)
        for result in by_problem_algorithm:
            ax.step(result.fevals, result["y"], where="post")
            ax.set_xscale("log")
            ax.set_xlabel("Number of function evaluation")
            ax.set_ylabel("Objective value")
    fig.tight_layout()
    

Instead, we want to look at runtime performance plots...

For an RTP plot, we need to tell cocoviz the indicator (important in the multi-objective case) we are interested in and if that indicator should be maximized or minimized.

In [ ]:
ind = Indicator("y", larger_is_better=False)

rtpplot(results, ind)

A similar check exists for the number of objectives.

So let's visualize by dimension and see how the algorithms perform.

In [ ]:
for dim, by_dim in results.by_number_of_variables():
    ax = rtpplot(by_dim, ind)
    ax.set_title(f"Dimension: {dim}")

Remove simulated annealing without local search because it looks inferior to simulated annealing with local search in most cases.

In [ ]:
no_sa = results.filter(lambda x: x.algorithm != "sa")

Now drill down by dimension and problem

In [ ]:
for problem, by_prob in no_sa.by_problem_name():
    fig, axes = plt.subplots(ncols=len(by_prob.number_of_variables), figsize=(12, 4), sharex=True)
    fig.suptitle(f"Problem: {problem}")
    for ax, (dim, by_dim_prob) in zip(axes, by_prob.by_number_of_variables()):
        ax.set_title(f"Dimension: {dim}")
        rtpplot(by_dim_prob, ind, ax=ax)
    fig.tight_layout()

## Advanced topics

### Custom targets

cocoviz tries hard to pick good defaults, but sometimes you want more control and you can change most of them.
Let's explore some of the options you have.

In [ ]:
from cocoviz.targets import linear_targets, log_targets

lin_targets = linear_targets(no_sa, Indicator("y", larger_is_better=False))
log_targets = log_targets(no_sa, Indicator("y", larger_is_better=False))

lin_targets

In [ ]:
for problem, by_prob in no_sa.by_problem_name():
    for dim, by_dim_prob in by_prob.by_number_of_variables():
        fig, axs = plt.subplots(ncols=2, figsize=(8, 4))
        fig.suptitle(f"Problem: {problem} / Dimension: {dim}")
        rtpplot(by_dim_prob, Indicator("y", larger_is_better=False), targets=lin_targets, ax=axs[0])        
        axs[0].set_title("linear targets")
    
        rtpplot(by_dim_prob, Indicator("y", larger_is_better=False), targets=log_targets, ax=axs[1])        
        axs[1].set_title("log targets")
        fig.tight_layout()

### Registering Indicators

Instead of having to pass the same `Indicator` object to repeated calls of `rtpplot`, we can register the indicator once, and then use the name (as a string) instead of the object.

In [ ]:
import cocoviz.indicator

cocoviz.indicator.register(Indicator("y", larger_is_better=False))

Once registered, we can use the string name of the performance indicator instead of passing the `Indicator` object.

In [ ]:
for problem, by_prob in results.by_problem_name():
    for dim, by_dim_prob in by_prob.by_number_of_variables():        
        ax = rtpplot(by_dim_prob, "y")                        
        ax.set_title(f"{problem} / d={dim}")

### Reading COCO datasets

If you have cocopp installed, you can import COCO datasets directly as `ResultSet`s and visualize them.


In [ ]:
!pip install cocopp

from cocoviz import read_coco_dataset

cma = read_coco_dataset("2020/CMA-ES-2019_Hansen.tgz")
powell = read_coco_dataset("2019/Powell-scipy-2019_Varelas.tgz")
bfgs = read_coco_dataset("2019/BFGS-scipy-2019_Varelas.tgz")
nm = read_coco_dataset("2019/Nelder-Mead-scipy-2019_Varelas.tgz")

combined = ResultSet()
combined.extend(cma)
combined.extend(powell)
combined.extend(bfgs)
combined.extend(nm)
combined

In [ ]:
combined[0]._data

Let's only look at the results on function $f_{21}$ by subsetting `combined` on the `problem_name` of each result.

In [ ]:
subset = combined.filter(lambda r: r.problem.name == "bbob-f_21")
coco_ind = Indicator("value", larger_is_better=False)

For COCO, we know that the value recorded is the difference in function value to the minimum so we can construct more meaningful targets

In [ ]:
import numpy as np

coco_targets = {}
for problem in subset.problems:
    coco_targets[problem] = np.logspace(2, -12)
    
next(iter(coco_targets.items()))

Now plot the results on $f_{21}$ by dimension using our COCO-like targets.

In [ ]:
for dim, by_dim in subset.by_number_of_variables():
    ax = rtpplot(by_dim, coco_ind, targets=coco_targets)
    ax.set_title(f"{dim} D")